# Minimum information tiers: metabolomics

How many detected metabolites (LC-MS features) of the Brassicaceae pilot reach each tier of the metabolite annotation, and what do the first hits look like?

**Case:** case 5, metabolomics: metabolite annotations. **Data:** annotation (evidence graphs of the pilots, `shape_constraints/rdf/` and `examples/`). **Queries:** the files of [`sparql-examples-pmw`](https://github.com/pathway-lod/sparql-examples-pmw/tree/main/examples/MinimumInformationTiers) (<https://pathway-lod.github.io/sparql-examples-pmw/>), run in a local Oxigraph store, with PlantMetWiki, Rhea and UniProt called by `SERVICE` on the live endpoints.

The outputs are a **dated snapshot**: the date and the PlantMetWiki release are in the summary table at the end. Each query is run once with a time limit; a tier that did not finish is shown as *timeout*. The results of every tier are also saved as CSV in `results/`. To rerun, open the notebook with a kernel that has the packages of `requirements.txt` installed (see the README of this folder).

In [1]:
import mi
import pandas as pd
from IPython.display import display
pd.set_option('display.max_colwidth', 80)
CASE = '5 metabolomics'
print('PlantMetWiki release:', mi.pmw_release())

PlantMetWiki release: r17.0.0_20260605-171052


## The data

In [2]:
data = ['annotation']
print('features with at least one annotation:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> SELECT (COUNT(DISTINCT ?f) AS ?n) WHERE { ?a a lodp:MetaboliteAnnotation ; lodp:feature ?f }'))
print('annotations:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> SELECT (COUNT(?a) AS ?n) WHERE { ?a a lodp:MetaboliteAnnotation }'))

features with at least one annotation: 2430


annotations: 2722


## Annotation matches PlantMetWiki

`annotation-matches-plantmetwiki`

A detected metabolite (feature) is annotated by at least one tool with an InChIKey whose first block (connectivity) equals that of a PlantMetWiki metabolite. The match is computed here, by comparing the first InChIKey block of each annotation with the PlantMetWiki metabolites; no stored anchor or match term is used. Returns, per feature, the number of tools whose annotation matches. TESTED 2026-11-10: on the 2,722 annotations of the Brassicaceae pilot it returned 2,015 features with a match, in about 63 s.

*Keywords:* InChIKey, Minimum Information: evidence tiers, PlantMetWiki metabolite, annotation tool, case 5: metabolomics, metabolite annotations, evidence tier, federated query, local evidence graph, metabolite annotation, metabolite feature

In [3]:
r = mi.run('annotation-matches-plantmetwiki', ['annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

rows | 2015 rows | 4.0 s


,feature,toolsMatching,exampleMetabolite
0,https://example.org/lod-pathways/pilot/feature_MTBLS14270_pos_4416,5,http://rdf-plantmetwiki.bioinformatics.nl/Pathway/PC1027_r17.0.0_20260605-17...
1,https://example.org/lod-pathways/pilot/feature_MTBLS14270_pos_3187,5,http://rdf-plantmetwiki.bioinformatics.nl/Pathway/PC1027_r17.0.0_20260605-17...
2,https://example.org/lod-pathways/pilot/feature_MTBLS14270_pos_681,5,http://rdf-plantmetwiki.bioinformatics.nl/Pathway/PC1072_r17.0.0_20260605-17...
3,https://example.org/lod-pathways/pilot/feature_MTBLS14270_neg_1823,5,http://rdf-plantmetwiki.bioinformatics.nl/Pathway/PC190_r17.0.0_20260605-171...
4,https://example.org/lod-pathways/pilot/feature_MTBLS14270_neg_819,5,http://rdf-plantmetwiki.bioinformatics.nl/Pathway/PC1043_r17.0.0_20260605-17...
5,https://example.org/lod-pathways/pilot/feature_MTBLS14270_pos_1036,5,http://rdf-plantmetwiki.bioinformatics.nl/Pathway/PC165_r17.0.0_20260605-171...
6,https://example.org/lod-pathways/pilot/feature_MTBLS14270_pos_2309,5,http://rdf-plantmetwiki.bioinformatics.nl/Pathway/PC190_r17.0.0_20260605-171...
7,https://example.org/lod-pathways/pilot/feature_MTBLS14270_pos_2734,5,http://rdf-plantmetwiki.bioinformatics.nl/Pathway/PC190_r17.0.0_20260605-171...
8,https://example.org/lod-pathways/pilot/feature_MTBLS14270_neg_392,5,http://rdf-plantmetwiki.bioinformatics.nl/Pathway/PC1043_r17.0.0_20260605-17...
9,https://example.org/lod-pathways/pilot/feature_MTBLS14270_pos_3445,5,http://rdf-plantmetwiki.bioinformatics.nl/Pathway/PC190_r17.0.0_20260605-171...


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Two metabolites share a chemical class

`shared-chemical-class`

Two detected metabolites of the same dataset, each with an annotation that matches PlantMetWiki (`annotation-matches-plantmetwiki`), share a chemical class (an NPClassifier term). The class is read from the annotation as lodp:chemicalClass. NOT RUNNABLE ON THE CURRENT GRAPHS: the annotation graph has no class values yet (NPClassifier is a step of the consensus workflow, but its output is not in the RDF).

*Keywords:* InChIKey, Minimum Information: evidence tiers, NPClassifier, PlantMetWiki metabolite, case 5: metabolomics, metabolite annotations, chemical class, evidence tier, federated query, local evidence graph, metabolite annotation, metabolite feature, no data yet

In [4]:
r = mi.run('shared-chemical-class', ['annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 0.0 s


,featureA,featureB,class


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Two metabolites linked by a similarity metric

`similarity-link`

Two detected metabolites are linked by a similarity metric (for example the cosine score of the MS/MS spectra, or an edge of a molecular network) above a threshold. The link is read as lodp:similarTo with lodp:similarityScore. NOT RUNNABLE ON THE CURRENT GRAPHS: no feature-to-feature similarity is in the RDF yet.

*Keywords:* MS/MS spectral similarity, Minimum Information: evidence tiers, case 5: metabolomics, metabolite annotations, evidence tier, local evidence graph, metabolite feature, molecular network, no data yet, similarity metric

In [5]:
r = mi.run('similarity-link', ['annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 0.0 s


,featureA,featureB,metric,score


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Two metabolites linked by a structure similarity (InChI)

`structure-similarity`

Proxy for a structure similarity. Two detected metabolites of the same dataset are matched to PlantMetWiki metabolites (`annotation-matches-plantmetwiki`) and the two InChIs share the same formula layer but not the same connectivity (they are related structures: isomers). A true fingerprint similarity (for example Tanimoto) cannot be computed in SPARQL; when precomputed values are in the graph (as in `similarity-link`) they replace this proxy. TESTED 2026-11-10: did NOT finish within 280 s (two unbound federated calls to PlantMetWiki); it needs a LIMIT, a cached lookup or the PlantMetWiki metabolites loaded locally.

*Keywords:* InChI, InChIKey, Minimum Information: evidence tiers, PlantMetWiki metabolite, case 5: metabolomics, metabolite annotations, evidence tier, federated query, local evidence graph, metabolite annotation, metabolite feature, structure similarity

In [6]:
r = mi.run('structure-similarity', ['annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

timeout | None rows | None s


""


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## The difference between two metabolites is a Rhea reaction with a PlantMetWiki enzyme

`rhea-step-between-metabolites`

Two detected metabolites are matched to PlantMetWiki metabolites (`annotation-matches-plantmetwiki`) and the difference between them is exactly one Rhea reaction: the two ChEBI compounds are on opposite sides of a Rhea reaction (federated query to Rhea), and PlantMetWiki has a reaction with that Rhea identifier and a UniProt enzyme. A hit suggests a pathway step between the two detected metabolites that PlantMetWiki may not have. Shows up to 100 pairs. TESTED 2026-11-10: did NOT finish within 280 s (three unbound federated calls and a join over Rhea); run it on a subset of features or with the lookups cached.

*Keywords:* ChEBI, InChIKey, Minimum Information: evidence tiers, PlantMetWiki metabolite, PlantMetWiki reaction, Rhea, UniProt, case 2: inferred enzyme function from PMN/PlantCyc+ db, case 5: metabolomics, metabolite annotations, enzyme, evidence tier, federated query, local evidence graph, metabolite annotation, metabolite feature, pathway expansion

In [7]:
r = mi.run('rhea-step-between-metabolites', ['annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

error: killed by the system (probably out of memory) | None rows | None s


""


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Summary of this case

In [8]:
s = pd.read_csv('results/summary.csv')
display(s[s['case'] == CASE].reset_index(drop=True))

,tier,case,status,n_rows,seconds,timeout_s,date,pmw_release
0,annotation-matches-plantmetwiki,5 metabolomics,rows,2015.0,4.0,150,2026-11-10,r17.0.0_20260605-171052
1,rhea-step-between-metabolites,5 metabolomics,error: killed by the system (probably out of memory),NaN,NaN,150,2026-11-10,r17.0.0_20260605-171052
2,shared-chemical-class,5 metabolomics,no rows,0.0,0.0,150,2026-11-10,r17.0.0_20260605-171052
3,similarity-link,5 metabolomics,no rows,0.0,0.0,150,2026-11-10,r17.0.0_20260605-171052
4,structure-similarity,5 metabolomics,timeout,NaN,NaN,150,2026-11-10,r17.0.0_20260605-171052
